## Step 7: Gene Prediction with BRAKER3
**Input:** Soft-masked genome from Step 6; 
fungal protein database from OrthoDB (odb12v2)  
**Output:** Gene models (GTF), predicted protein sequences (`braker.aa`) 
in `09-annot/prot_only/`  
**Tools:** BRAKER3 (GeneMark-EP v4.72 + AUGUSTUS v3.5.0); 
OrthoDB odb12v2 (Fungi subset via selectClade.py)  
**Key parameters:** Protein evidence only; fungal branch point model; 
new species model training  
**Key finding:** 11,738 genes predicted; 37,626 exons (~3.2 exons/gene); 
12,547 CDS entries  
**Reference:** Materials & Methods Section 5 — Nebli et al. (2025)

## Prepare Data for Annotation

### Set up

In [ ]:
export SN="3RR" \
NCPUS=48 \
NJOB=8 \
NCPUS_PER_JOB=16 \
MASKED_ASSEMBLY="07-repeats-analysis/rm_out/${SN}.fasta.masked" \
WORK_ASSEMBLY="08-data/assembly.fa" \
PROTEINS="$PWD/08-data/orthodb/Fungi.fa"

cp "$MASKED_ASSEMBLY" "08-data/${SN}.fasta.masked" && \
awk '/^>/{print ">chr" ++i; next} {print}' \
"08-data/${SN}.fasta.masked" > "$WORK_ASSEMBLY" && \
export assembly="$PWD/$WORK_ASSEMBLY"

### Create Fungi Protein Database from OrthoDB

In [ ]:
mkdir -p 08-data/orthodb

wget https://data.orthodb.org/v12/download/odb_data_dump/odb12v2_aa_fasta.gz -O 08-data/orthodb/odb12v2_aa_fasta.gz
wget https://data.orthodb.org/v12/download/odb_data_dump/odb12v2_levels.tab.gz -O 08-data/orthodb/odb12v2_levels.tab.gz
wget https://data.orthodb.org/v12/download/odb_data_dump/odb12v2_level2species.tab.gz -O 08-data/orthodb/odb12v2_level2species.tab.gz
gunzip 08-data/orthodb/*.gz

curl -L https://raw.githubusercontent.com/tomasbruna/orthodb-clades/master/selectClade.py -o 08-data/orthodb/selectClade.py
chmod +x 08-data/orthodb/selectClade.py

In [ ]:
./A-08-data/orthodb/selectClade.py \
  A-08-data/orthodb/odb12v2_aa_fasta \
  A-08-data/orthodb/odb12v2_levels.tab \
  A-08-data/orthodb/odb12v2_level2species.tab \
  Fungi > 08-data/orthodb/Fungi.fa

python3 A-08-data/orthodb/selectClade.py \
  A-08-data/orthodb/odb12v2_aa_fasta \
  A-08-data/orthodb/odb12v2_levels.tab \
  A-08-data/orthodb/odb12v2_level2species.tab \
  Fungi > 08-data/orthodb/Fungi.fa

# BRAKER3

In [ ]:
alias braker.pl="apptainer run --writable-tmpfs docker://teambraker/braker3:v3.0.7.6 braker.pl"

In [ ]:
mkdir -p 09-annot/prot_only
braker.pl \
    --genome=08-data/assembly.fa \
    --prot_seq=$proteins \
    --species=${SN}_prot \
    --workingdir=09-annot/prot_only \
    --threads=$NCPUS \
    --fungus

# count genes 

In [ ]:
#Count unique gene IDs
grep -P "\tgene\t" 09-annot/prot_only/braker.gtf \
| sed 's/.*gene_id "\(.*\)".*/\1/' \
| sort -u \
| wc -l
#11738
#Count gene features
grep -P "\tgene\t" 09-annot/prot_only/braker.gtf | wc -l
#11738

#for exons count
grep -P "\texon\t" 09-annot/prot_only/braker.gtf | wc -l
#37626
grep -c ">" 09-annot/prot_only/braker.aa
#12547

# --------------------------
# using Braker output

# Functional Annotation

In [ ]:
tmux new -s FunAn_script
tmux attach -t FunAn_script
tmux kill-session -t FunAn_script

In [ ]:
export SN=sample10
export NCPUS=128
export proteome="$PWD/09-annot/prot_only/braker.aa"

# Note: The path to the proteome file assumes it was generated by BRAKER in the previous chapter.
# Adjust the path if your file is located elsewhere.

In [ ]:
alias interproscan_app="apptainer run docker://interpro/interproscan:5.75-106.0"
alias eggnog-mapper="apptainer run docker://quay.io/biocontainers/eggnog-mapper:2.1.13--pyhdfd78af_0"

mkdir -p 10-functional-annotation/interproscan
curl -o 10-functional-annotation/interproscan/interproscan-data-5.75-106.0.tar.gz http://ftp.ebi.ac.uk/pub/software/unix/iprscan/5/5.75-106.0/alt/interproscan-data-5.75-106.0.tar.gz

tar -pxzf 10-functional-annotation/interproscan/interproscan-data-5.75-106.0.tar.gz -C 10-functional-annotation/interproscan

# Run InterProScan

In [ ]:
mkdir -p 10-functional-annotation/interproscan/{input,temp,output}
cp $proteome 10-functional-annotation/interproscan/input/
cp $genome 10-functional-annotation/interproscan/input/


# We need to bind the data, input, temp, and output directories to the container.
# Note that we are using the full path to the directories.
DATADIR=$PWD/10-functional-annotation/interproscan/interproscan-5.75-106.0/data
INPUTDIR=$PWD/10-functional-annotation/interproscan/input
TEMPDIR=$PWD/10-functional-annotation/interproscan/temp
OUTPUTDIR=$PWD/10-functional-annotation/interproscan/output

# We build a full command with bind mounts instead of using the simple alias
# apptainer run \
#     --bind $DATADIR:/opt/interproscan/data \
#     --bind $INPUTDIR:/input \
#     --bind $TEMPDIR:/temp \
#     --bind $OUTPUTDIR:/output \
#     docker://interpro/interproscan:5.75-106.0 \
#     --input /input/$(basename $proteome) \
#     --output-dir /output \
#     --tempdir /temp \
#     --cpu $NCPUS \
#     --goterms \
#     --pathways

apptainer exec \
    --bind "$DATADIR":/opt/interproscan/data \
    --bind "$INPUTDIR":/input \
    --bind "$TEMPDIR":/temp \
    --bind "$OUTPUTDIR":/output \
    ~/interpro_tmp/interproscan_5.75-106.0.sif \
    /opt/interproscan/interproscan.sh \
    --input /input/proteome_clean.fasta \
    --output-dir /output \
    --tempdir /temp \
    --cpu "$NCPUS" \
    --goterms \
    --pathways

# if the interproscan image didnt work, use below

In [ ]:
# mkdir -p ~/interpro_tmp
# mkdir -p ~/interpro_cache

# export APPTAINER_TMPDIR=~/interpro_tmp
# export APPTAINER_CACHEDIR=~/interpro_cache
# export TMPDIR=~/interpro_tmp

# apptainer pull \
# ~/interpro_tmp/interproscan_5.75-106.0.sif \
# docker://interpro/interproscan:5.75-106.0

In [ ]:
Option 2 — Build or download yourself (if you have permission)

You can obtain antiSMASH container images from official sources.

Example workflow:

apptainer pull antismash.sif docker://antismash/antismash

Then test:

apptainer exec antismash.sif antismash --help

In [ ]:
singularity pull antismash.sif docker://antismash/standalone
singularity run antismash.sif antismash assembly.fa
# apptainer pull antismash.sif docker://antismash/standalone
# alias antismash="apptainer run docker://antismash/standalone antismash"

In [ ]:
# conda install -c bioconda agat
conda create -n genome_annot python=3.10
conda activate genome_annot
conda install -c bioconda agat
singularity pull agat.sif docker://quay.io/biocontainers/agat:1.0.0--pl5321hdfd78af_0

# gffread 09-annot/prot_only/braker.gtf -g 08-data/assembly.fa -F -o 10-functional-annotation/genome.gbk   

#Converts the annotation into true GFF3 format
gffread 09-annot/prot_only/braker.gtf -T -o braker_tmp.gff3
gffread braker_tmp.gff3 -o braker.gff3


In [ ]:
gffread 09-annot/prot_only/braker.gtf -T -o braker_tmp.gff3
gffread braker_tmp.gff3 -o braker.gff3

singularity exec antismash.sif antismash \
08-data/assembly.fa \
--taxon fungi \
--genefinding-tool none \
--genefinding-gff3 braker.gff3 \
--output-dir 10-functional-annotation/antismash_out \
--cpus 32

# For richer annotation

In [ ]:
singularity exec antismash.sif antismash \
08-data/assembly.fa \
--taxon fungi \
--genefinding-tool none \
--genefinding-gff3 braker.gff3 \
--output-dir 10-functional-annotation/antismash_out_richer \
--cpus 32 \
--enable-nrps-pks \
--enable-t2pks \
--enable-terpene \
--enable-lanthipeptides \
--enable-lassopeptides \
--enable-thiopeptides \
--enable-sactipeptides \
--enable-tta \
--enable-genefunctions \
--enable-html

# EggNog-mapper

In [ ]:
# mkdir -p 10-functional-annotation/eggnog

# # Download the main eggnog databases
# # eggnog-mapper download_eggnog_data.py -y --data_dir 10-functional-annotation/eggnog

# export EGGNOG_DATA_DIR=/fpool/opt/db/eggnog-data


# eggnog-mapper create_dbs.py \
# -m diamond \
# --dbname fungi \
# --taxa Fungi \
# --data_dir $EGGNOG_DATA_DIR


# # Create a diamond database for Fungi.
# # This will download all eggNOG proteins first if not present, then create the fungi-specific database.
# # eggnog-mapper create_dbs.py -m diamond --dbname fungi --taxa Fungi --data_dir 10-functional-annotation/eggnog

In [ ]:
# Step 1 — Create your own writable eggNOG directory
# # mkdir -p ~/eggnog_data
# Step 2 — Copy the required base DB files (since you have read access)
# cp /fpool/opt/db/eggnog-data/eggnog.db 10-functional-annotation/eggnog
# cp /fpool/opt/db/eggnog-data/eggnog_proteins.dmnd 10-functional-annotation/eggnog
# cp /fpool/opt/db/eggnog-data/eggnog.taxa.db* 10-functional-annotation/eggnog

# (This copies metadata but not rebuilding from scratch.)

# Step 3 — Point eggNOG to your directory
# export EGGNOG_DATA_DIR=~/eggnog_data
# Step 4 — Now create fungi database
# eggnog-mapper create_dbs.py \
# -m diamond \
# --dbname fungi \
# --taxa Fungi \
# --data_dir $EGGNOG_DATA_DIR

In [ ]:
mkdir -p 10-functional-annotation/eggnog/output

DATADIR=/fpool/opt/db/eggnog-data
INPUTFILE=$proteome
OUTPUTDIR=$PWD/10-functional-annotation/eggnog/output
OUTPUT_PREFIX=${SN}_eggnog

apptainer run \
    --bind $DATADIR:/data \
    --bind $(dirname $INPUTFILE):/input \
    --bind $OUTPUTDIR:/output \
    docker://quay.io/biocontainers/eggnog-mapper:2.1.13--pyhdfd78af_0 \
    emapper.py \
    -i /input/$(basename $INPUTFILE) \
    -o $OUTPUT_PREFIX \
    --output_dir /output \
    --data_dir /data \
    --dmnd_db /data/fungi.dmnd \
    --cpu $NCPUS \
    --override

# sample10_eggnog.emapper.annotations

In [ ]:
head -20 sample10_eggnog.emapper.annotations
grep -v "^#" sample10_eggnog.emapper.annotations | wc -l
cut -f1,8 sample10_eggnog.emapper.annotations | column -t | head
cut -f1,12 sample10_eggnog.emapper.annotations | head


In [ ]:
1️⃣ Count metabolism-related genes
grep -v "^#" sample10_eggnog.emapper.annotations | grep -w "G" | wc -l
# 644
2️⃣ Extract all enzymes (EC numbers present)
awk -F'\t' '$11 != "-" {print $1, $11}' sample10_eggnog.emapper.annotations
3️⃣ Extract CAZy enzymes (important for fungi)
awk -F'\t' '$14 != "-" {print $1, $14}' sample10_eggnog.emapper.annotations
Count Most Frequent Modules

Run:

cut -f14 sample10_eggnog.emapper.annotations \
| tr ',' '\n' \
| sort | uniq -c | sort -nr

In [ ]:
🧬 Step 1 — Count Total Annotated Genes
grep -v "^#" sample10_eggnog.emapper.annotations | wc -l

This gives total annotated proteins.

📊 Step 2 — Summary Per Database

We summarize each annotation field separately.

1️⃣ GO Terms Summary
Count proteins with GO terms
awk -F'\t' '$10 != "-" {count++} END {print "Proteins with GO:", count}' sample10_eggnog.emapper.annotations
Count total GO terms
awk -F'\t' '$10 != "-" {print $10}' sample10_eggnog.emapper.annotations \
| tr ',' '\n' | sort | uniq | wc -l
2️⃣ EC Numbers Summary
Count enzyme proteins
awk -F'\t' '$11 != "-" {count++} END {print "Proteins with EC:", count}' sample10_eggnog.emapper.annotations
Enzyme class distribution
awk -F'\t' '$11 != "-" {print $11}' sample10_eggnog.emapper.annotations \
| tr ',' '\n' \
| cut -d'.' -f1 \
| sort | uniq -c | sort -nr

This shows how many oxidoreductases, hydrolases, etc.

3️⃣ KEGG KO Summary
Proteins with KO
awk -F'\t' '$12 != "-" {count++} END {print "Proteins with KO:", count}' sample10_eggnog.emapper.annotations
Unique KO count
awk -F'\t' '$12 != "-" {print $12}' sample10_eggnog.emapper.annotations \
| tr ',' '\n' | sort | uniq | wc -l
4️⃣ KEGG Pathway Summary
awk -F'\t' '$13 != "-" {print $13}' sample10_eggnog.emapper.annotations \
| tr ',' '\n' | sort | uniq -c | sort -nr
5️⃣ KEGG Module Summary
awk -F'\t' '$14 != "-" {print $14}' sample10_eggnog.emapper.annotations \
| tr ',' '\n' | sort | uniq -c | sort -nr
6️⃣ COG Category Summary
awk -F'\t' '$7 != "-" {print $7}' sample10_eggnog.emapper.annotations \
| fold -w1 \
| sort | uniq -c | sort -nr

This gives functional category distribution.

7️⃣ PFAM Domains Summary
awk -F'\t' '$21 != "-" {print $21}' sample10_eggnog.emapper.annotations \
| tr ',' '\n' | sort | uniq -c | sort -nr | head
8️⃣ CAZy Summary (Very Important for Fungi)
awk -F'\t' '$15 != "-" {print $15}' sample10_eggnog.emapper.annotations \
| tr ',' '\n' | sort | uniq -c | sort -nr

# Interproscna


In [ ]:
#Count total protein entries
grep -c "^##sequence-region" proteome_clean.fasta.gff3
11664

awk '$1 !~ /^##/ {print $2}' proteome_clean.fasta.gff3 | sort | uniq -c | sort -nr
#Breakdown by database/module
awk '$1!~/^#/ && $2!="." {print $2}' proteome_clean.fasta.gff3 | sort | uniq -c | sort -nr
577904
  33319 MobiDBLite
  14333 Pfam
  12951 Gene3D
  10330 SUPERFAMILY
   9204 PANTHER
   7397 PIRSR
   6359 PRINTS
   5656 SMART
   5541 ProSiteProfiles
   4819 CDD
   3980 FunFam
   3161 Coils
   2457 ProSitePatterns
   1172 NCBIfam
    792 PIRSF
    351 Hamap
    202 SFLD

# fbouzid@ngsserver:~/SN/10-functional-annotation/interproscan/output$ grep -v "^#" proteome_clean.fasta.gff3 | cut -f9 | grep -oP 'Name=[^;]+' | sort | uniq -c | sort -nr | head -20
#   33319 Name=mobidb-lite
#    3161 Name=Coil
#     672 Name=SM00248
#     596 Name=PR00081
#     593 Name=G3DSA:3.40.50.300
#     570 Name=SM00320
#     540 Name=SSF52540
#     491 Name=G3DSA:1.20.1250.20
#     475 Name=G3DSA:3.40.50.720
#     466 Name=cd00067
#     453 Name=SSF57701
#     444 Name=G3DSA:4.10.240.10
#     432 Name=SSF51735
#     431 Name=PS50048
#     413 Name=SSF103473
#     410 Name=SM00066
#     404 Name=PF00172
#     378 Name=PS00463
#     348 Name=cd12148
#     279 Name=SSF53474

fbouzid@ngsserver:~/SN/10-functional-annotation/interproscan/output$ awk -F'\t' '
  $0 !~ /^#/ {
    if($9 ~ /Ontology_term=/){
      print $1
    }
  }
' proteome_clean.fasta.gff3 | sort | uniq | wc -l
6730



import re
from collections import Counter

gff_file = "proteome_clean.fasta.gff3"

pfam_counts = Counter()

with open(gff_file) as f:
    for line in f:
        if line.startswith('#'):
            continue
        fields = line.strip().split('\t')
        if len(fields) < 9:
            continue
        attributes = fields[8]
        # Extract Pfam domain from Name=
        pfams = re.findall(r'Name=(PF\d+)', attributes)
        pfam_counts.update(pfams)

# Total Pfam hits
total_hits = sum(pfam_counts.values())
print(f"Total Pfam hits: {total_hits}")

# Number of unique Pfam domains
print(f"Unique Pfam domains: {len(pfam_counts)}\n")

# Top 20 Pfam domains by count
print("Top 20 Pfam domains:")
for pfam, count in pfam_counts.most_common(20):
    print(f"{pfam}: {count}")


# Unique Pfam domains: 4416
# PF00172 : 404
# PF04082 : 272
# PF00400: 258
# PF07690: 254
# PF12796: 156
# PF00153: 134
# PF20684: 122
# PF00069: 121
# PF00083: 116
# PF11951: 111
# PF00106: 105
# PF00067: 103
# PF00076: 94
# PF00005: 83
# PF00664: 71
# PF00096: 70
# PF08240: 69
# PF00501: 60
# PF00668: 58
# PF00107: 56

# Create a list of top 20 Pfam IDs
cat > top20_pfams.txt <<EOL
PF00172
PF04082
PF00400
PF07690
PF12796
PF00153
PF20684
PF00069
PF00083
PF11951
PF00106
PF00067
PF00076
PF00005
PF00664
PF00096
PF08240
PF00501
PF00668
PF00107
EOL

# Extract signature_desc for each Pfam
while read pfam; do
    grep "Name=$pfam" proteome_clean.fasta.gff3 | \
    sed -n 's/.*signature_desc=\([^;]*\);.*/\1/p' | \
    head -1
done < top20_pfams.txt

Fungal Zn(2)-Cys(6) binuclear cluster domain
Fungal specific transcription factor domain
WD domain, G-beta repeat
Major Facilitator Superfamily
Ankyrin repeats (3 copies)
Mitochondrial carrier protein
Fungal rhodopsin domain
Protein kinase domain
Sugar (and other) transporter
Fungal specific transcription factor domain
short chain dehydrogenase
Cytochrome P450
RNA recognition motif
ABC transporter
ABC transporter transmembrane region
Zinc finger, C2H2 type
Alcohol dehydrogenase GroES-like domain
AMP-binding enzyme
Condensation domain
Zinc-binding dehydrogenase




In [ ]:
wget http://purl.obolibrary.org/obo/go/go-basic.obo -O go-basic.obo

import re
from collections import defaultdict
from goatools.obo_parser import GODag

# 1️⃣ Load your annotation file
gff_file = "proteome_clean.fasta.gff3"

# 2️⃣ Prepare a GO dictionary to count terms by category
go_counts = defaultdict(int)

# 3️⃣ Load the GO ontology (you need go-basic.obo from GO)
# Download from: http://purl.obolibrary.org/obo/go/go-basic.obo
go_obo = "go-basic.obo"
godag = GODag(go_obo)

# 4️⃣ Extract GO terms from the GFF
with open(gff_file) as f:
    for line in f:
        if line.startswith('#'):
            continue
        attrs = line.strip().split('\t')[-1]  # last column: attributes
        # Look for GO terms in Dbxref, e.g., Dbxref="InterPro:IPR036663","GO:0003824"
        match = re.findall(r'GO:(\d{7})', attrs)
        for go_id in match:
            go_id_full = f"GO:{go_id}"
            if go_id_full in godag:
                go_term = godag[go_id_full]
                category = go_term.namespace  # mf / bp / cc
                go_counts[category] += 1

# 5️⃣ Print the counts
print("GO term counts by category:")
print(f"Molecular Function: {go_counts['molecular_function']}")
print(f"Biological Process: {go_counts['biological_process']}")
print(f"Cellular Component: {go_counts['cellular_component']}")

GO term counts by category:
>>> print(f"Molecular Function: {go_counts['molecular_function']}")
Molecular Function: 37616
>>> print(f"Biological Process: {go_counts['biological_process']}")
Biological Process: 14488
>>> print(f"Cellular Component: {go_counts['cellular_component']}")
Cellular Component: 4526


In [ ]:
# 1️⃣ Total number of proteins in your annotation file (excluding header)
awk 'NR>4 {print $1}' sample10_eggnog.emapper.annotations | wc -l
# This counts all query proteins annotated by EggNOG

# 2️⃣ Number of proteins with at least one EggNOG functional annotation
# Column 6 usually contains COG category, column positions may vary; adjust if needed
awk 'NR>4 && $6!="" {print $1}' sample10_eggnog.emapper.annotations | wc -l
# Gives proteins with COG assignment

# 3️⃣ % of proteins without annotation
TOTAL=$(awk 'NR>4 {print $1}' sample10_eggnog.emapper.annotations | wc -l)
ANNOTATED=$(awk 'NR>4 && $6!="" {print $1}' sample10_eggnog.emapper.annotations | wc -l)
echo "Unannotated %: " $(( (TOTAL - ANNOTATED)*100/TOTAL ))

# 4️⃣ Distribution of COG functional categories
awk 'NR>4 && $6!="" {print $6}' sample10_eggnog.emapper.annotations | tr ',' '\n' | sort | uniq -c | sort -nr
# Lists COG categories (like J, K, L) and counts

# 5️⃣ Number of proteins with GO terms
awk 'NR>4 && $9!="" {print $1}' sample10_eggnog.emapper.annotations | wc -l
# Column 9 usually contains GO terms

# 6️⃣ Number of proteins with KEGG pathway annotations
awk 'NR>4 && $10!="" {print $1}' sample10_eggnog.emapper.annotations | wc -l
# Column 10 usually contains KEGG pathways

# 7️⃣ Number of proteins with eggNOG ortholog assignments
awk 'NR>4 && $5!="" {print $1}' sample10_eggnog.emapper.annotations | wc -l
# Column 5 usually contains eggNOG ortholog

In [ ]:
import pandas as pd

# Input file
input_file = "sample10_eggnog.emapper.annotations"

# Read the file, skipping comment lines starting with ##
df = pd.read_csv(input_file, sep="\t", comment="#", low_memory=False)

# List of annotation columns to extract
annotations = {
    "KEGG": ["KEGG_ko", "KEGG_Pathway", "KEGG_Module", "KEGG_Reaction", "KEGG_rclass"],
    "COG": ["COG_category", "eggNOG_OGs", "max_annot_lvl", "Description"],
    "PFAM": ["PFAMs"],
    "CAZy": ["CAZy"],
    "GOs": ["GOs"],
    "EC": ["EC"],
    "BRITE": ["BRITE"],
    "TC": ["KEGG_TC"],
    "BiGG": ["BiGG_Reaction"]
}

# Create output CSV files for each annotation type
for name, cols in annotations.items():
    # Keep only query and relevant columns
    subset = df[["query"] + [c for c in cols if c in df.columns]]
    
    # Filter rows with non-empty values
    subset = subset.dropna(how="all", subset=cols)
    
    # Save to CSV
    subset.to_csv(f"{name}_annotations.csv", index=False)
    print(f"[+] Saved {name}_annotations.csv with {len(subset)} entries")